# Quantify scratchpadLocal sandbox for the AmplifyME momentum work. `quantify` isn't installed here, so theharness falls back to `amp.engine_shim` — a stand-in that mirrors the real engine'sdocumented behaviour, including the one-order-per-ticker-per-bar rule and the balancecheck.

In [ ]:
import sys, ossys.path.insert(0, os.getcwd())sys.path.insert(0, os.path.join(os.getcwd(), "challenge"))sys.path.insert(0, os.path.join(os.getcwd(), "tests"))import numpy as np, pandas as pdimport matplotlib.pyplot as pltpd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)import run_local as Rfrom dual_panel import dual_panelprint("ready")

## Synthetic dataThree generators, each with a verified statistical signature.

In [ ]:
from amp.features import reversion_mappanels = {    "trending":  R.daily_panel(n=350, kind="momentum", seed=3),    "reverting": R.daily_panel(n=350, kind="reverting", seed=3),    "dual":      dual_panel(seed=3, drift_vol=0.008, rev_strength=0.016, rev_phi=0.88),}for name, px in panels.items():    m = reversion_map(px, lookbacks=(5,20,60), horizons=(5,10,20)).stack().mean()    print("%-10s shape %s   mean xs-autocorr %+.3f" % (name, px.shape, m))

## Run the submitted strategyEdit the parameters at the top of `challenge/submit_v8.py`, re-run this cell.

In [ ]:
import importlib.utildef load(path, name="strat"):    spec = importlib.util.spec_from_file_location(name, path)    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)    return m.momentum_strategydef backtest(strategy, prices, capital=R.CAPITAL, commission=R.COMMISSION, quiet=True):    """Mirror of the challenge runner, with the real engine's constraints enforced."""    import io, contextlib    engine = R.StrictEngine(data=prices, starting_balance=capital,                            commission_percentage=commission)    curve = []    buf = io.StringIO()    with (contextlib.redirect_stdout(buf) if quiet else contextlib.nullcontext()):        for i in range(len(prices)):            strategy(engine, prices.iloc[:i+1], prices.index[i])            engine._mark_all(prices.index[i])            curve.append(capital + engine.profit_loss)    eq = pd.Series(curve, index=prices.index)    r = eq.pct_change().dropna(); active = r[r != 0]    return {        "pnl": engine.profit_loss,        "return_pct": engine.profit_loss / capital * 100,        "sharpe": float(r.mean()/r.std()*np.sqrt(252)) if r.std() > 0 else np.nan,        "max_dd_pct": float((eq/eq.cummax() - 1).min() * 100),        "win_rate_pct": float((active > 0).mean()*100) if len(active) else np.nan,        "trades": engine.get_total_trade_count(),        "equity": eq,        "log": buf.getvalue(),    }strat = load("challenge/submit_v8.py")res = backtest(strat, panels["trending"])print({k: (round(v,2) if isinstance(v,float) else v) for k,v in res.items() if k not in ("equity","log")})

## Compare versions side by side

In [ ]:
versions = {    "v8  (no quality)":  "challenge/submit_v8.py",    "v9  (quality 1.0)": "challenge/submit_v9.py",    "v10 (quality 0.5)": "challenge/submit_v10.py",}rows = []fig, ax = plt.subplots(figsize=(11,5))for label, path in versions.items():    try:        out = backtest(load(path, label), panels["trending"])    except FileNotFoundError:        continue    rows.append({"version": label, **{k: out[k] for k in                 ("return_pct","sharpe","max_dd_pct","win_rate_pct","trades")}})    (out["equity"]/R.CAPITAL - 1).mul(100).plot(ax=ax, label=label)ax.set_ylabel("return %"); ax.legend(); ax.grid(alpha=0.3); ax.set_title("trending panel")display(pd.DataFrame(rows).round(2))

## Regime table for any panelThe same diagnostic the strategy prints in the challenge terminal.

In [ ]:
print(reversion_map(panels["dual"]).round(3).to_string())